In [ ]:
import os
import shutil

output_dir = "/workspace/PyTorchSim/outputs"

# Delete previous outputs if they exist to prevent unintended reuse of stale results
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

In [ ]:
import os
import sys
import torch
import torch._dynamo
import torch.utils.cpp_extension
base_dir = os.environ.get('TORCHSIM_DIR', default='/workspace/PyTorchSim')
os.environ['TORCHSIM_LOG_PATH']=os.path.join(os.getcwd(), "togsim_results")
sys.path.append(base_dir)

device = torch.device("npu:0")

def test_result(name, npu_output, cpu_output, rtol=1e-4, atol=1e-4, preview=5):
    npu_output_cpu = npu_output.cpu()

    if torch.allclose(npu_output_cpu, cpu_output, rtol=rtol, atol=atol):
        message = f"|{name} Functionality Test Passed|"
        print("-" * len(message))
        print(message)
        print("-" * len(message))

        npu_flat = npu_output_cpu.flatten()
        cpu_flat = cpu_output.flatten()

        npu_preview = ", ".join(f"{x.item():.3f}" for x in npu_flat[:preview])
        cpu_preview = ", ".join(f"{x.item():.3f}" for x in cpu_flat[:preview])

        print(f"npu output: [{npu_preview}, ...]")
        print(f"cpu output: [{cpu_preview}, ...]")

    else:
        message = f"|{name} Functionality Test Failed|"
        print("-" * len(message))
        print(message)
        print("-" * len(message))
        print("npu output:", npu_output_cpu)
        print("cpu output:", cpu_output)
        exit(1)

def test_exponent2(device, size=(128, 128)):
    def exponent2(a):
        return a.exp2()
    x = torch.randn(size).to(device=device)
    opt_fn = torch.compile(dynamic=False)(exponent2)
    res = opt_fn(x)
    out = exponent2(x.cpu())
    test_result("exponent2", res, out)

In [ ]:
# os.environ['TOGSIM_CONFIG']=f"{base_dir}/tutorial/session1/togsim_configs/togsim_config_functional_only.yml" 

input = torch.randn(16, 16)
npu_x = input.to(device=device)
cpu_x = input.to("cpu")
func = torch.exp2
opt_fn = torch.compile(dynamic=False)(func)
npu_out = opt_fn(npu_x)
cpu_out = func(cpu_x)
test_result("exp2", npu_out, cpu_out)

In [ ]:
!cat /workspace/PyTorchSim/outputs/.torchinductor/f2/cf2vu2l7lk4enrgroufjilktl6vfmvnfhrt33dp2nku2bohoipjd.py